In [13]:
!pip install -q faiss-cpu gradio

In [14]:
import numpy as np
import json
import faiss
import time
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('Gemini_API_Key'))

In [72]:
resumes = [

{
    "name":"Adhiti Sharma",
    "skills":["Python","FastAPI","Docker","PostgreSQL"],
    "experience":"Software Engineering Intern at Flipkart",
    "education":"BTech CSE IIT Delhi"
},

{
    "name":"Rahul Verma",
    "skills":["Java","Spring Boot","MySQL","AWS"],
    "experience":"Backend Developer at TCS for 3 years",
    "education":"BTech CSE VIT"
},

{
    "name":"Priya Reddy",
    "skills":["React","JavaScript","HTML","CSS"],
    "experience":"Frontend Developer at Infosys",
    "education":"BTech IT"
},

{
    "name":"Arjun Nair",
    "skills":["TensorFlow","PyTorch","Machine Learning","Python"],
    "experience":"AI Engineer at Startup",
    "education":"MTech AI"
},

{
    "name":"Sneha Gupta",
    "skills":["Recruitment","Communication","HRMS"],
    "experience":"HR Executive at Wipro",
    "education":"MBA HR"
},

{
    "name":"Karan Singh",
    "skills":["Digital Marketing","SEO","Content Writing"],
    "experience":"Marketing Associate",
    "education":"MBA Marketing"
},

{
    "name":"Neha Kapoor",
    "skills":["Accounting","Excel","GST","Tally"],
    "experience":"Accountant at Deloitte",
    "education":"BCom"
},

{
    "name":"Vikram Rao",
    "skills":["Sales","Negotiation","CRM"],
    "experience":"Sales Manager",
    "education":"MBA"
},

{
    "name":"Pooja Mehta",
    "skills":["Python","Data Analysis","SQL","Power BI"],
    "experience":"Data Analyst at Accenture",
    "education":"BTech"
},

{
    "name":"Ravi Kumar",
    "skills":["C++","Embedded Systems","Arduino"],
    "experience":"Embedded Engineer",
    "education":"BTech ECE"
},

{
"name":"Rahul",
"skills":["Python","Machine Learning","NLP","TensorFlow"],
"education":"B.Tech CSE",
"experience":"2 years ML Engineer"
},

{
"name":"Priya",
"skills":["Java","Spring Boot","MySQL"],
"education":"B.Tech IT",
"experience":"3 years Backend Developer"
},

{
"name":"Arjun",
"skills":["React","JavaScript","HTML","CSS"],
"education":"B.Tech CSE",
"experience":"Frontend Developer"
},

{
"name":"Sneha",
"skills":["Data Science","Python","Pandas","Statistics"],
"education":"M.Tech AI",
"experience":"Data Analyst"
},

{
"name":"Vikram",
"skills":["AWS","Docker","Kubernetes"],
"education":"B.Tech CSE",
"experience":"DevOps Engineer"
},

{
"name":"Meena",
"skills":["Recruitment","HR","Hiring"],
"education":"MBA HR",
"experience":"HR Executive"
},

{
"name":"Anjali",
"skills":["Marketing","SEO","Content Writing"],
"education":"MBA Marketing",
"experience":"Marketing Specialist"
},

{
"name":"Ravi",
"skills":["Sales","CRM","Negotiation"],
"education":"B.Com",
"experience":"Sales Executive"
},

{
"name":"Pooja",
"skills":["Accounting","GST","Finance"],
"education":"M.Com",
"experience":"Accountant"
},

{
"name":"Harish",
"skills":["Cyber Security","Network Security"],
"education":"B.Tech",
"experience":"Security Analyst"
},

{
"name":"Akash",
"skills":["Android","Kotlin","Java"],
"education":"B.Tech",
"experience":"Android Developer"
},

{
"name":"Nisha",
"skills":["Business Analysis","SQL","Excel"],
"education":"MBA",
"experience":"Business Analyst"
},

{
"name":"Ramesh",
"skills":["Product Management","Agile"],
"education":"MBA",
"experience":"Product Manager"
},

{
"name":"Kiran",
"skills":["Python","Deep Learning","Computer Vision"],
"education":"B.Tech",
"experience":"AI Engineer"
},

{
"name":"Divya",
"skills":["C++","DSA","Competitive Programming"],
"education":"B.Tech",
"experience":"Software Engineer"
},

{
"name":"Varun",
"skills":["Cloud","Azure","DevOps"],
"education":"B.Tech",
"experience":"Cloud Engineer"
},

{
"name":"Sai",
"skills":["Python","NLP","LLMs","Generative AI"],
"education":"B.Tech",
"experience":"GenAI Engineer"
},

{
"name":"Bhavana",
"skills":["Teaching","Communication"],
"education":"M.Sc",
"experience":"Lecturer"
},

{
"name":"Lokesh",
"skills":["Electrical Systems","PLC"],
"education":"B.Tech EEE",
"experience":"Electrical Engineer"
},

{
"name":"Keerthi",
"skills":["Embedded Systems","Arduino","C"],
"education":"B.Tech ECE",
"experience":"Embedded Engineer"
}

]

In [17]:
def create_profile(resume):

    return f"""
    Name: {resume['name']}
    Skills: {', '.join(resume['skills'])}
    Education: {resume['education']}
    Experience: {resume['experience']}
    """

In [50]:
def get_embedding(text):

    result = genai.embed_content(
        model="models/gemini-embedding-001",
        content=text,
        task_type="retrieval_query"
    )

    return np.array(
        result["embedding"],
        dtype=np.float32
    )

In [19]:
vectors = []
metadata = []

for resume in resumes:

    profile = create_profile(resume)

    embedding = get_embedding(profile)

    vectors.append(embedding)

    metadata.append(resume)

vectors = np.array(vectors)

np.save(
    "resumes.npy",
    vectors
)

with open(
    "resumes_meta.json",
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

print("Embeddings Saved")

Embeddings Saved


In [21]:
vectors = np.load("resumes.npy").astype("float32")
faiss.normalize_L2(vectors)
dimension = vectors.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(vectors)

faiss.write_index(
    index,
    "faiss_index.bin"
)

print("FAISS Index Saved")

FAISS Index Saved


In [35]:
def search_candidates_numpy(
    jd,
    top_k=5
):

    vectors = np.load(
        "resumes.npy"
    )

    jd_embedding = get_embedding(jd)

    scores = np.dot(
        vectors,
        jd_embedding
    ) / (

        np.linalg.norm(
            vectors,
            axis=1
        ) *

        np.linalg.norm(
            jd_embedding
        )
    )

    idx = np.argsort(
        scores
    )[::-1][:top_k]

    results = []

    for i in idx:

        if scores[i] < 0.55:
            continue

        results.append(
            (
                metadata[i]["name"],
                float(scores[i])
            )
        )

    if results==None:
      return "No strong matches found"
    else:
      return results

In [36]:
def search_candidates_faiss(
    jd,
    top_k=5
):

    index = faiss.read_index("faiss_index.bin")

    query = get_embedding(jd)

    query = np.array([query]).astype("float32")

    faiss.normalize_L2(query)

    scores, ids = index.search(
        query,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        ids[0]
    ):

        if score < 0.55:
            continue

        results.append(
            (
                metadata[idx]["name"],
                float(score)
            )
        )

    return results

In [51]:
jd = """
Looking for a Python Developer
with Machine Learning,
NLP and Deep Learning experience.
"""

In [52]:
start = time.time()

numpy_results = search_candidates_numpy(jd)

numpy_time = time.time() - start

print("NUMPY RESULTS")

for r in numpy_results:
    print(r)

print("\nTime:", numpy_time)

NUMPY RESULTS
('Sai', 0.7064926028251648)
('Rahul', 0.7034180760383606)
('Arjun Nair', 0.6931241154670715)
('Kiran', 0.6911702156066895)
('Sneha', 0.6774865984916687)

Time: 1.521270990371704


In [53]:
start = time.time()

faiss_results = search_candidates_faiss(jd)

faiss_time = time.time() - start

print("FAISS RESULTS")

for r in faiss_results:
    print(r)

print("\nTime:", faiss_time)

FAISS RESULTS
('Sai', 0.70649254322052)
('Rahul', 0.7034181356430054)
('Arjun Nair', 0.6931241154670715)
('Kiran', 0.6911702752113342)
('Sneha', 0.6774866580963135)

Time: 1.038074016571045


In [54]:
print(
    "Same Rankings:",
    numpy_results == faiss_results
)

Same Rankings: False


In [55]:
jd2 = """
Looking for an HR recruiter
with talent acquisition
and hiring experience.
"""

print(
    search_candidates_numpy(jd2)
)

[('Meena', 0.7295582890510559), ('Sneha Gupta', 0.7139281630516052), ('Neha Kapoor', 0.6727508306503296), ('Pooja', 0.6639772057533264), ('Bhavana', 0.6639650464057922)]


In [56]:
jd3 = """
Alien spaceship banana quantum
dragon icecream football moon.
"""

print(
    search_candidates_faiss(jd3)
)

[('Bhavana', 0.6393667459487915), ('Keerthi', 0.6266404390335083), ('Anjali', 0.6249168515205383), ('Divya', 0.62473064661026), ('Varun', 0.6232542991638184)]


In [110]:
genai.configure(
    api_key=userdata.get('Gemini_API_Key_4')
)
model= genai.GenerativeModel(
    model_name="gemini-2.5-flash",
)

In [111]:
def explain_matches(jd, candidates):

    candidate_text = ""

    for i, candidate in enumerate(candidates, start=1):

        candidate_text += f"""
Candidate {i}
Name: {candidate['name']}
Skills: {', '.join(candidate['skills'])}
Education: {candidate['education']}
Experience: {candidate['experience']}
"""

    prompt = f"""
Job Description:
{jd}

For each candidate, give a one-line explanation
(maximum 15 words).

{candidate_text}
"""

    response = model.generate_content(prompt)

    return response.text.strip()

In [114]:
results = search_candidates_faiss(jd)

selected_candidates = []

for name, score in results:

    candidate = next(
        r for r in metadata
        if r["name"] == name
    )

    selected_candidates.append(candidate)

print(
    explain_matches(
        jd,
        selected_candidates
    )
)

Here's a one-line explanation for each candidate:

**Candidate 1: Sai**
Excellent fit: Python, strong NLP and Deep Learning (GenAI/LLMs) experience.

**Candidate 2: Rahul**
Strong fit: Python, Machine Learning, NLP, and Deep Learning (TensorFlow) experience.

**Candidate 3: Arjun Nair**
Good fit: Strong ML/DL (TensorFlow/PyTorch) and Python, but NLP experience not explicit.

**Candidate 4: Kiran**
Good Python and Deep Learning; however, lacks explicit Machine Learning and NLP.

**Candidate 5: Sneha**
Limited fit: Python and Data Science skills, lacks specific ML/NLP/DL experience.


In [112]:
results = search_candidates_faiss(jd2)

selected_candidates = []

for name, score in results:

    candidate = next(
        r for r in metadata
        if r["name"] == name
    )

    selected_candidates.append(candidate)

print(
    explain_matches(
        jd2,
        selected_candidates
    )
)

Here are the one-line explanations for each candidate:

**Candidate 1: Meena**
Excellent fit; possesses direct recruitment, HR, and hiring experience.

**Candidate 2: Sneha Gupta**
Strong fit with recruitment experience as an HR Executive.

**Candidate 3: Neha Kapoor**
Not suitable; experience is in accounting, not HR recruitment.

**Candidate 4: Pooja**
Not suitable; experience is in accounting and finance.

**Candidate 5: Bhavana**
Not suitable; background is in teaching, not HR or recruitment.


In [113]:
results = search_candidates_faiss(jd3)

selected_candidates = []

for name, score in results:

    candidate = next(
        r for r in metadata
        if r["name"] == name
    )

    selected_candidates.append(candidate)
print("JD:",jd3)
print(
    explain_matches(
        jd3,
        selected_candidates
    )
)

JD: 
Alien spaceship banana quantum
dragon icecream football moon.

The job description "Alien spaceship banana quantum dragon icecream football moon" is wonderfully nonsensical! Therefore, the explanations will reflect this absurdity.

Here are the one-line explanations for each candidate:

**Candidate 1: Bhavana**
Teaching and communication skills don't align with alien quantum banana dragons.

**Candidate 2: Keerthi**
Embedded systems expertise won't help with quantum dragon banana footballs.

**Candidate 3: Anjali**
Could market alien banana quantum dragons, ice cream, or even the moon.

**Candidate 4: Divya**
Strong programming skills are irrelevant for alien spaceship banana quantum dragons.

**Candidate 5: Varun**
Cloud and DevOps expertise won't manage quantum dragons or moon infrastructure.
